Name: Nadya Renata Siahaan, Alena Destria Pramesti
NIM : 093, 254
Class : 2025INT
Exercise 6 Pemteks — Feature Engineering

Pada tugas ini, semua teknik feature engineering yang sudah diajarkan di kelas diterapkan pada **hasil preprocessing Week 5** (teks dari website `http://home.wezm.net`):

1. One Hot Encoding
2. Bag of Words (CountVectorizer)
3. TF-IDF
4. Word2Vec (+ visualisasi UMAP)
5. FastText (+ visualisasi UMAP)

## Setup

In [1]:
!pip install gensim umap-learn

In [2]:
import warnings
warnings.filterwarnings(action='ignore')

import re
import string
import requests
import numpy as np
import pandas as pd
import plotly.express as px
from bs4 import BeautifulSoup

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

## Preprocessing (dari Week 5)

Di Week 5, seluruh teks website diproses menjadi **satu string panjang**. Feature engineering membutuhkan **beberapa dokumen** supaya bisa dibandingkan (misalnya TF-IDF menghitung seberapa jarang kata muncul di dokumen lain). Karena itu, teks website dipecah per **kalimat**, lalu setiap kalimat diproses dengan langkah yang sama seperti Week 5: lower casing → hapus tanda baca, angka, dan simbol → hapus stopwords. Setiap kalimat hasil preprocessing menjadi satu dokumen.

In [3]:
# MENGAMBIL SEMUA TEKS (sama seperti Week 5)
webrequest = requests.get("http://home.wezm.net")
soup = BeautifulSoup(webrequest.text, "html.parser")

paragraphs = soup.find_all('p')
raw_text = " ".join(p.text for p in paragraphs)
print(raw_text)


                  Welcome to my retro website! It's about some retro things and implemented with retro tech
                  like HTML4 and tables. You can read more about how it was built in this post on my blog:
                  Building a Website Fit for 1999.
                 
                  This site is available
                  over https for modern browsers,
                  and http for retro browsers.
                 Live energy stats for the server: ¹ Since 2025-04-15T00:00:00 Other websites that serve as inspiration or have things in common with this one:


In [4]:
# MEMECAH TEKS MENJADI KALIMAT (setiap kalimat = 1 dokumen)
kalimat = sent_tokenize(raw_text)
for i, k in enumerate(kalimat):
    print(i, ':', " ".join(k.split()))

0 : Welcome to my retro website!
1 : It's about some retro things and implemented with retro tech like HTML4 and tables.
2 : You can read more about how it was built in this post on my blog: Building a Website Fit for 1999.
3 : This site is available over https for modern browsers, and http for retro browsers.
4 : Live energy stats for the server: ¹ Since 2025-04-15T00:00:00 Other websites that serve as inspiration or have things in common with this one:


In [5]:
# FUNGSI PREPROCESSING (langkah-langkah dari Week 5)
punc = string.punctuation
sw = set(stopwords.words('english'))

def preprocessing(text):
    # lower casing
    text = text.lower()
    # remove punctuations, numbers, and symbols
    text = text.translate(str.maketrans('', '', punc))
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    # remove stopwords
    return " ".join([word for word in text.split() if word not in sw and len(word) > 1])

df = pd.DataFrame({'text': kalimat})
df['prepro'] = df['text'].apply(preprocessing)
df

,text,prepro
0,\n Welcome to my retro website!,welcome retro website
1,It's about some retro things and implemented w...,retro things implemented retro tech like html ...
2,You can read more about how it was built in th...,read built post blog building website fit
3,This site is available\n over...,site available https modern browsers http retr...
4,Live energy stats for the server: ¹ Since 2025...,live energy stats server since websites serve ...


In [6]:
# SIMPAN HASIL PREPROCESSING
# (di Week 5 error karena hasil_st berupa string; di sini disimpan sebagai DataFrame)
df[['prepro']].to_csv("hasil_preprocessing.csv", index=False)

In [7]:
# LOAD HASIL PREPROCESSING
df = pd.read_csv('hasil_preprocessing.csv')
df = df[df['prepro'].str.strip() != ''].dropna().reset_index(drop=True)
df_pre = df['prepro']
df_pre.info()

<class 'pandas.core.series.Series'>
RangeIndex: 5 entries, 0 to 4
Series name: prepro
Non-Null Count  Dtype 
--------------  ----- 
5 non-null      object
dtypes: object(1)
memory usage: 172.0+ bytes


In [8]:
df_pre

0                                welcome retro website
1    retro things implemented retro tech like html ...
2            read built post blog building website fit
3    site available https modern browsers http retr...
4    live energy stats server since websites serve ...
Name: prepro, dtype: object

## 1. One Hot Encoding

In [9]:
vectorizer_ohe = CountVectorizer(binary=True)
X_ohe = vectorizer_ohe.fit_transform(df_pre)
df_ohe = pd.DataFrame(X_ohe.toarray(), columns=vectorizer_ohe.get_feature_names_out())
df_ohe

,available,blog,browsers,building,built,common,energy,fit,html,http,...,server,since,site,stats,tables,tech,things,website,websites,welcome
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1
1,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,1,1,1,0,0,0
2,0,1,0,1,1,0,0,1,0,0,...,0,0,0,0,0,0,0,1,0,0
3,1,0,1,0,0,0,0,0,0,1,...,0,0,1,0,0,0,0,0,0,0
4,0,0,0,0,0,1,1,0,0,0,...,1,1,0,1,0,0,1,0,1,0


In [10]:
print('Jumlah dokumen          :', df_ohe.shape[0])
print('Jumlah fitur (kata unik):', df_ohe.shape[1])
print('Nilai di dalam matriks  :', np.unique(X_ohe.toarray()))

Jumlah dokumen          : 5
Jumlah fitur (kata unik): 31
Nilai di dalam matriks  : [0 1]


## 2. Bag of Words (CountVectorizer)

In [11]:
# Mengubah teks menjadi numerik
vectorizer_bow = CountVectorizer()
X_bow = vectorizer_bow.fit_transform(df_pre)
df_bow = pd.DataFrame(X_bow.toarray(), columns=vectorizer_bow.get_feature_names_out())
df_bow

,available,blog,browsers,building,built,common,energy,fit,html,http,...,server,since,site,stats,tables,tech,things,website,websites,welcome
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1
1,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,1,1,1,0,0,0
2,0,1,0,1,1,0,0,1,0,0,...,0,0,0,0,0,0,0,1,0,0
3,1,0,2,0,0,0,0,0,0,1,...,0,0,1,0,0,0,0,0,0,0
4,0,0,0,0,0,1,1,0,0,0,...,1,1,0,1,0,0,1,0,1,0


In [12]:
# 10 kata yang paling sering muncul di seluruh dokumen
df_bow.sum().sort_values(ascending=False).head(10)

retro        4
website      2
browsers     2
things       2
building     1
available    1
blog         1
fit          1
html         1
http         1
dtype: int64

## 3. TF-IDF

In [13]:
# Mengubah teks menjadi matriks TF-IDF
vectorizer_tfidf = TfidfVectorizer()
X_tfidf = vectorizer_tfidf.fit_transform(df_pre)
df_FE = pd.DataFrame(X_tfidf.toarray(), columns=vectorizer_tfidf.get_feature_names_out())
df_FE.round(3)

,available,blog,browsers,building,built,common,energy,fit,html,http,...,server,since,site,stats,tables,tech,things,website,websites,welcome
0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.557,0.000,0.69
1,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.366,0.000,...,0.000,0.000,0.000,0.000,0.366,0.366,0.296,0.000,0.000,0.00
2,0.000,0.388,0.000,0.388,0.388,0.000,0.000,0.388,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.313,0.000,0.00
3,0.325,0.000,0.651,0.000,0.000,0.000,0.000,0.000,0.000,0.325,...,0.000,0.000,0.325,0.000,0.000,0.000,0.000,0.000,0.000,0.00
4,0.000,0.000,0.000,0.000,0.000,0.306,0.306,0.000,0.000,0.000,...,0.306,0.306,0.000,0.306,0.000,0.000,0.247,0.000,0.306,0.00


In [14]:
# Kata dengan bobot TF-IDF tertinggi di setiap dokumen
for i in range(len(df_FE)):
    top = df_FE.iloc[i].sort_values(ascending=False).head(3)
    print(f"Dokumen {i}:", ", ".join(f"{w} ({v:.3f})" for w, v in top.items()))

Dokumen 0: welcome (0.690), website (0.557), retro (0.462)
Dokumen 1: retro (0.491), implemented (0.366), like (0.366)
Dokumen 2: blog (0.388), built (0.388), building (0.388)
Dokumen 3: browsers (0.651), available (0.325), https (0.325)
Dokumen 4: energy (0.306), common (0.306), live (0.306)


## Tokenize Sentence

In [15]:
from nltk import word_tokenize
from tqdm import tqdm

sentences = [word_tokenize(doc.lower()) for doc in tqdm(df_pre)]
sentences

100%|██████████| 5/5 [00:00<00:00, 1250.46it/s]


[['welcome', 'retro', 'website'],
 ['retro', 'things', 'implemented', 'retro', 'tech', 'like', 'html', 'tables'],
 ['read', 'built', 'post', 'blog', 'building', 'website', 'fit'],
 ['site',
  'available',
  'https',
  'modern',
  'browsers',
  'http',
  'retro',
  'browsers'],
 ['live',
  'energy',
  'stats',
  'server',
  'since',
  'websites',
  'serve',
  'inspiration',
  'things',
  'common',
  'one']]

## 4. Word2Vec

**Penjelasan:** Word2Vec adalah *word embedding*, yaitu teknik yang mengubah setiap kata menjadi **vektor angka padat (dense)** berdasarkan konteks kata-kata di sekitarnya. Kata yang sering muncul dalam konteks yang mirip akan memiliki vektor yang berdekatan. Berbeda dengan One Hot/BoW/TF-IDF yang ukuran vektornya bergantung pada jumlah vocabulary, ukuran vektor Word2Vec tetap (di sini 100 dimensi).

In [16]:
import gensim.models
# 1. Train Model
model = gensim.models.Word2Vec(sentences, min_count=1, vector_size=100, window=5, sg=0)

param
- `min_count=1`: Includes words that appear at least once.
- `vector_size=100`: Generates word vectors of 100 dimensions.
- `window=5`: Considers a context window of 5 words before and after the target word.
- `sg=0`: Uses CBOW model (default setting).
- `sg=1`: Enables the Skip-Gram model (predicts context words given a target word).

In [17]:
# 2. Save Model
model.save('week5.w2v')

In [18]:
# 3. Load Model
model = gensim.models.Word2Vec.load('week5.w2v')

In [19]:
# 4. Model Information
w2v = model.wv
w2v.index_to_key[:10]

['retro',
 'browsers',
 'things',
 'website',
 'one',
 'common',
 'inspiration',
 'serve',
 'websites',
 'since']

In [20]:
w2v.vectors.shape, w2v.vector_size # size of vector

((31, 100), 100)

In [21]:
# Kata yang paling mirip dengan 'retro'
w2v.similar_by_key('retro')

[('https', 0.2188381403684616),
 ('since', 0.21614821255207062),
 ('tech', 0.1954425573348999),
 ('welcome', 0.15160661935806274),
 ('like', 0.09937148541212082),
 ('energy', 0.09309972822666168),
 ('serve', 0.09280624985694885),
 ('building', 0.08403972536325455),
 ('stats', 0.07960055768489838),
 ('blog', 0.06438063830137253)]

In [22]:
# 5. FE result
# Vektor dokumen = rata-rata vektor semua kata di dalam dokumen
def get_document_vector(tokens, model):
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not valid_vectors:
        return np.zeros(model.vector_size)
    return np.mean(valid_vectors, axis=0)

X_features_w2v = np.array([get_document_vector(doc, model) for doc in sentences])
df_w2v = pd.DataFrame(X_features_w2v)
df_w2v

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.002861,0.003894,0.001991,0.004877,-0.003041,-0.003629,0.005403,0.006055,0.001764,-0.002324,...,-0.001933,0.000533,0.002712,-0.000244,0.009706,0.002773,-0.003409,-0.005298,-0.003324,-0.001494
1,0.000986,0.000532,-0.002099,0.003124,0.000779,0.001101,0.002114,0.001311,-0.003257,0.002952,...,0.000221,0.001540,0.002727,-0.002732,0.002623,0.000470,-0.001887,-0.001861,0.000455,0.002321
2,-0.000919,0.002748,0.003138,0.002649,-0.003952,-0.000493,0.000651,0.001499,0.001221,-0.003648,...,0.000210,-0.003019,-0.003132,0.001091,0.000248,-0.003642,0.001903,0.000361,-0.000839,0.001939
3,-0.000554,0.001048,0.002477,0.001859,0.003705,-0.004661,0.002117,0.006075,-0.001848,-0.003452,...,0.003875,0.000819,0.003387,-0.000668,0.000590,0.003579,0.000103,-0.000158,-0.002185,0.003636
4,-0.000797,0.000978,0.000036,-0.000217,-0.000463,-0.000305,0.000288,0.001625,-0.003687,-0.001829,...,0.001484,0.001727,-0.000094,0.000095,0.003322,-0.000548,0.002658,-0.000569,0.002643,-0.000377


### Visualize Word2Vec

In [23]:
   %pip install --upgrade nbformat

Note: you may need to restart the kernel to use updated packages.


In [24]:
from umap import UMAP

X = UMAP(n_neighbors=5, random_state=42).fit_transform(w2v.vectors)

df2 = pd.DataFrame(X, columns=['umap1', 'umap2'])
df2['word'] = w2v.index_to_key

fig = px.scatter(df2, x='umap1', y='umap2', text='word')
fig.update_traces(textposition='top center')
fig.update_layout(height = 800,
                  title_text = 'Word2Vec Visualization')
fig.show()

## 5. FastText

In [25]:
import gensim.models
# 1. build model
model_fasttext = gensim.models.FastText(sentences, min_count=1,
                                        vector_size=100, window=5,
                                        min_n=3, max_n=6,
                                        sg=1, epochs=10)

In [26]:
# 2. save model
model_fasttext.save('week5.fasttext')

In [27]:
# 3. load model
model_fasttext = gensim.models.FastText.load('week5.fasttext')

In [28]:
# 4. model information
fastext = model_fasttext.wv
fastext.index_to_key[:10]

['retro',
 'browsers',
 'things',
 'website',
 'one',
 'common',
 'inspiration',
 'serve',
 'websites',
 'since']

In [29]:
fastext.vectors.shape, fastext.vector_size # size of vector

((31, 100), 100)

In [30]:
# Kata yang paling mirip dengan 'retro'
fastext.similar_by_key('retro')

[('tables', 0.19445572793483734),
 ('live', 0.18942801654338837),
 ('read', 0.13580526411533356),
 ('building', 0.13545651733875275),
 ('available', 0.12475880235433578),
 ('stats', 0.09932195395231247),
 ('server', 0.05458337441086769),
 ('serve', 0.04852424934506416),
 ('website', 0.03616618737578392),
 ('tech', 0.03549918159842491)]

In [31]:
print("'websit' ada di vocabulary Word2Vec?", 'websit' in w2v.key_to_index)
print("'websit' ada di vocabulary FastText?", 'websit' in fastext.key_to_index)
fastext.similar_by_key('websit')

'websit' ada di vocabulary Word2Vec? False
'websit' ada di vocabulary FastText? False


[('website', 0.613968551158905),
 ('websites', 0.5977228283882141),
 ('retro', 0.20039957761764526),
 ('post', 0.15818031132221222),
 ('built', 0.11847162246704102),
 ('since', 0.11502809077501297),
 ('building', 0.11103862524032593),
 ('blog', 0.10256443917751312),
 ('energy', 0.08469918370246887),
 ('welcome', 0.0756256952881813)]

In [32]:
# 5. FE result
X_features_ft = np.array([get_document_vector(doc, model_fasttext) for doc in sentences])
df_fasttext = pd.DataFrame(X_features_ft)
df_fasttext

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,0.000230,0.000607,0.000442,-0.001520,-0.000041,-0.000576,-0.000570,0.000327,-0.000721,-0.000238,...,0.000591,-0.002301,0.000287,-0.000473,0.001654,0.000022,0.001388,0.000056,0.000639,0.000391
1,-0.000016,0.000325,-0.000528,-0.000865,-0.000579,-0.000275,-0.000019,-0.000676,-0.000743,0.000361,...,-0.000877,-0.000730,-0.000423,-0.000549,0.000972,0.000812,0.001053,-0.000221,0.000640,-0.000121
2,0.000850,0.000418,0.000278,-0.000504,-0.001185,0.000522,-0.000293,0.000476,0.000647,0.000167,...,0.000610,-0.000217,-0.000350,-0.000401,-0.000040,-0.001091,0.001033,-0.000739,-0.000167,0.000535
3,-0.000842,-0.000289,0.000412,-0.000147,0.000420,-0.000325,0.000961,-0.000125,0.000136,-0.000624,...,0.000589,-0.000917,0.000466,0.001069,0.000025,0.000577,0.000586,0.000500,-0.000066,0.000339
4,0.000284,0.000733,-0.000071,-0.000165,-0.000143,0.000382,-0.000803,0.000741,0.000307,-0.000078,...,-0.000089,0.000526,-0.000435,-0.000319,0.000421,-0.000174,0.000155,-0.000516,-0.000416,-0.000258


### Visualize FastText

In [33]:
X = UMAP(n_neighbors=5, random_state=42).fit_transform(fastext.vectors)

df2 = pd.DataFrame(X, columns=['umap1', 'umap2'])
df2['word'] = fastext.index_to_key

fig = px.scatter(df2, x='umap1', y='umap2', text='word')
fig.update_traces(textposition='top center')
fig.update_layout(height = 800,
                  title_text = 'fastext Visualization')
fig.show()